# Data Processing

## Set Config

In [1]:
# ライブラリのインポート
import os
import numpy as np
import polars as pl
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import seaborn as sns

In [2]:
# フォルダ設定
data_dir = '../data/'
output_dir = '../output/'

# date
date = '20260202_emergency'

#predicting delirium within
within = '72h'

# 特徴量tableのファイル名
features_file_name = f'{date}_features.csv'
df_development_file_name = f'{date}_df_development_raw.csv'

In [3]:
### パラメータ設定
# 予測区間の選択
X1_hour = 72 # predicting_timeが72hの時はここを変える
X2_hour = 12 

# withlagモデル用のパラメータ
sampling_withlag_unstable_highrisk = 20
sampling_withlag_unstable_lowrisk = 10
sampling_withlag_stable_lowrisk = 10

# withoutlagモデル用のパラメータ
sampling_withoutlag_unstable_highrisk = 20
sampling_withoutlag_unstable_lowrisk = 10
sampling_withoutlag_stable_lowrisk = 10

# baselineモデル用のパラメータ
sampling_baseline_unstable_highrisk = 20
sampling_baseline_unstable_lowrisk = 10
sampling_baseline_stable_lowrisk = 10

In [4]:
# 保存するファイル名の指定
df_development_train_sampled_baseline_file_name = f'{date}_df_development_{within}_train_sampled_baseline.csv'
df_development_test_sampled_baseline_file_name = f'{date}_df_development_{within}_test_sampled_baseline.csv'
df_development_train_sampled_without_lag_file_name = f'{date}_df_development_{within}_train_sampled_without_lag.csv'
df_development_test_sampled_without_lag_file_name = f'{date}_df_development_{within}_test_sampled_without_lag.csv'
df_development_train_sampled_with_lag_file_name = f'{date}_df_development_{within}_train_sampled_with_lag.csv'
df_development_test_sampled_with_lag_file_name = f'{date}_df_development_{within}_test_sampled_with_lag.csv'
df_development_train_all_file_name = f'{date}_df_development_{within}_train_all.csv'
df_development_test_all_file_name = f'{date}_df_development_{within}_test_all.csv'

## Load Data

In [5]:
# データの読み込み
df_development = pl.read_csv(
    os.path.join(data_dir + df_development_file_name), 
    columns=['icu_stay_id','time_window_index','label_delirium_indication', 'delirium_flag']  # メモリ節約のため、必要なカラムだけ読み込む
)

In [6]:
# record数と症例数を確認
print('Number of records (model development cohort):', len(df_development))
print('Number of icu_stay_id (model development cohort):', len(df_development['icu_stay_id'].unique()))
print('Average number of records per icu_stay_id (model development cohort):', len(df_development) / len(df_development['icu_stay_id'].unique()))

Number of records (model development cohort): 4899756
Number of icu_stay_id (model development cohort): 41300
Average number of records per icu_stay_id (model development cohort): 118.6381598062954


In [7]:
# データの最初を見てみる。読み込めているか確認
df_development.head()

icu_stay_id,time_window_index,delirium_flag,label_delirium_indication
i64,i64,i64,i64
1021325852177154048,21,0,0
1021325852177154048,22,0,0
1021325852177154048,23,0,0
1021325852177154048,24,0,0
1021325852177154048,25,0,0


## Calculate Sample Size

In [8]:
### df_development
# サンプルサイズを計算
sample_size = len(df_development['icu_stay_id'].unique())

# せん妄になった症例数を計算
number_of_delirium_patients = len(df_development.filter(pl.col('label_delirium_indication') == 1)['icu_stay_id'].unique())

# 上記で計算した症例数を表示
print('==== Model Development Cohort ====')
print(f'Sample size is: {sample_size}')
print(f'Number of Delirium indicated patients included: {number_of_delirium_patients} ({number_of_delirium_patients/sample_size*100:.2f}%)')

==== Model Development Cohort ====
Sample size is: 41300
Number of Delirium indicated patients included: 12321 (29.83%)


## Outcome Labeling

症例$k$のICU入室時間を$T_0$、ICU退室時間を$T_k$とする。
- 24h以内の予測では$X_1 = 24 $ , $X_2 = 12 $とした
- 72h以内の予測では$X_1 = 72 $ , $X_2 = 12 $とした

Outcome labelingの方針
- せん妄発症症例（label_delirium_indication = 1）：最初にせん妄を発症した時刻を$T_i$とする
    - $T_i < t$のデータは消去する（切り出しの時点で消去ずみ）
    - $T_i - X_1 \leqq t < T_i$をハイリスクとして定義する
    - $T_i - X_1 - X_2 \leqq t < T_i - X_1$をリスク不明として定義する
    - $0 \leqq t < T_i - X_1 - X_2$をローリスクとして定義する

- 不変症例（label_delirium_indication = 0）：
    -  $0 \leqq t < T_k$（全入室期間）を低リスクと定義する

In [9]:
def labeling_per_id( df_pid: pl.DataFrame, X1_hour: int, X2_hour: int) -> pl.DataFrame:

    X1 = X1_hour
    X2 = X2_hour

    # この症例が「一度でもせん妄あり」か（行内で矛盾が出ないよう max を見る）
    positive = df_pid.select(pl.max("label_delirium_indication")).item() == 1

    if positive:
        # 仕様：各 icu_stay_id で delirium_flag=1 は最大1行
        Ti = (
            df_pid
            .filter(pl.col("delirium_flag") == 1)
            .select("time_window_index")
            .item()                      # 1件だけのはずなので .item() で取得
        )

        # まず未割当（-2）で初期化
        df_pid = df_pid.with_columns(pl.lit(-2).alias("y"))

        # High: [Ti - X1, Ti)  … 直前 X1時間
        df_pid = df_pid.with_columns(
            pl.when(
                (pl.col("time_window_index") >= Ti - X1) &
                (pl.col("time_window_index") <  Ti)
            ).then(1).otherwise(pl.col("y")).alias("y")
        )

        # Low: (-∞, Ti - X1 - X2)
        df_pid = df_pid.with_columns(
            pl.when(
                pl.col("time_window_index") < Ti - X1 - X2
            ).then(0).otherwise(pl.col("y")).alias("y")
        )

        # Undefined: [Ti - X1 - X2, Ti - X1)
        df_pid = df_pid.with_columns(
            pl.when(
                (pl.col("time_window_index") >= Ti - X1 - X2) &
                (pl.col("time_window_index") <  Ti - X1)
            ).then(-1).otherwise(pl.col("y")).alias("y")
        )

        # （保険）学習に使うのは y∈{-1,0,1} のみ
        df_pid = df_pid.filter(pl.col("y").is_in([-1, 0, 1]))

    else:
        # せん妄なし症例は全ウィンドウ Low
        df_pid = df_pid.with_columns(pl.lit(0).alias("y"))

    return df_pid


In [10]:
# labeling_per_id関数をそれぞれの症例に対して適用する関数
def provide_outcome_label(df: pl.DataFrame, X1_hour: int, X2_hour: int) -> pl.DataFrame:
    
    df_output = df.group_by('icu_stay_id').map_groups(lambda group: labeling_per_id(group, X1_hour, X2_hour))

    return df_output

In [11]:
# リスクレベルを定義
df_development = provide_outcome_label(df_development, X1_hour=X1_hour, X2_hour=X2_hour)

## Train Test Split

In [12]:
# model development cohortをtrain dataとtest dataに分ける関数
def split_on_patients(df: pl.DataFrame) -> tuple[pl.DataFrame, pl.DataFrame]:
    unique_ids = df['icu_stay_id'].unique().to_numpy()

    # train_test_splitを使用して症例ベースでtrainとtestに分割
    train_ids, test_ids = train_test_split(unique_ids, test_size=0.2, random_state=710)

    # データフレームを分割
    train_df = df.filter(pl.col('icu_stay_id').is_in(train_ids))
    test_df = df.filter(pl.col('icu_stay_id').is_in(test_ids))

    return train_df, test_df

In [13]:
# せん妄発症患者とそうでない患者を分ける
df_development_stable = df_development.filter(pl.col('label_delirium_indication') == 0)
df_development_unstable = df_development.filter(pl.col('label_delirium_indication') == 1)

In [14]:
# trainとtestに分割
# df_development_train, df_development_test = split_on_patients(df_development)

# trainとtestに分割
df_development_stable_train, df_development_stable_test = split_on_patients(df_development_stable)
df_development_unstable_train, df_development_unstable_test = split_on_patients(df_development_unstable)

In [15]:
# test症例をconcat
df_development_test_all = pl.concat([df_development_unstable_test, df_development_stable_test])

In [16]:
# 症例数の確認
len(df_development_stable_train['icu_stay_id'].unique()), len(df_development_stable_test['icu_stay_id'].unique()), len(df_development_unstable_train['icu_stay_id'].unique()), len(df_development_unstable_test['icu_stay_id'].unique())

(23183, 5796, 9856, 2465)

## 時間のランダムサンプリング

In [17]:
def sample_time_up_to_x(group: pl.DataFrame, x) -> pl.DataFrame:
    if len(group) < x:
        return group
    else:
        return group.sample(n=x, seed=710)

In [18]:
def sample_data(df, x, constraint = None):
    sampled_df = (
        df.group_by('icu_stay_id') \
        .map_groups(lambda group: sample_time_up_to_x(group, x))
    )
    
    return sampled_df

In [19]:
# 関数を適用し時間のrandom samplingを行う（baselineモデル用）
df_development_unstable_train_highrisk_baseline = sample_data(
                    df_development_unstable_train.filter(pl.col('y')==1), 
                    sampling_baseline_unstable_highrisk,
                    constraint=None)
df_development_unstable_train_lowrisk_baseline = sample_data(
                    df_development_unstable_train.filter(pl.col('y')==0), 
                    sampling_baseline_unstable_lowrisk,
                    constraint=None)
df_development_stable_train_baseline = sample_data(
                    df_development_stable_train, 
                    sampling_baseline_stable_lowrisk,
                    constraint=None)

df_development_unstable_test_highrisk_baseline = sample_data(
                    df_development_unstable_test.filter(pl.col('y')==1), 
                    sampling_baseline_unstable_highrisk,
                    constraint=None)
df_development_unstable_test_lowrisk_baseline = sample_data(
                    df_development_unstable_test.filter(pl.col('y')==0), 
                    sampling_baseline_unstable_lowrisk,
                    constraint=None)
df_development_stable_test_baseline = sample_data(
                    df_development_stable_test, 
                    sampling_baseline_stable_lowrisk,
                    constraint=None)

In [20]:
# train, test dataframeの結合（lag特徴量を使わないモデル用）
df_development_train_sampled_baseline = pl.concat([df_development_unstable_train_highrisk_baseline, df_development_unstable_train_lowrisk_baseline, df_development_stable_train_baseline])
df_development_test_sampled_baseline = pl.concat([df_development_unstable_test_highrisk_baseline, df_development_unstable_test_lowrisk_baseline, df_development_stable_test_baseline])

In [21]:
# 関数を適用し時間のrandom samplingを行う（lag特徴量を使わないモデル用）
df_development_unstable_train_highrisk_without_lag = sample_data(
                    df_development_unstable_train.filter(pl.col('y')==1), 
                    sampling_withoutlag_unstable_highrisk,
                    constraint=None)
df_development_unstable_train_lowrisk_without_lag = sample_data(
                    df_development_unstable_train.filter(pl.col('y')==0), 
                    sampling_withoutlag_unstable_lowrisk,
                    constraint=None)
df_development_stable_train_without_lag = sample_data(
                    df_development_stable_train, 
                    sampling_withoutlag_stable_lowrisk,
                    constraint=None)

df_development_unstable_test_highrisk_without_lag = sample_data(
                    df_development_unstable_test.filter(pl.col('y')==1), 
                    sampling_withoutlag_unstable_highrisk,
                    constraint=None)
df_development_unstable_test_lowrisk_without_lag = sample_data(
                    df_development_unstable_test.filter(pl.col('y')==0), 
                    sampling_withoutlag_unstable_lowrisk,
                    constraint=None)
df_development_stable_test_without_lag = sample_data(
                    df_development_stable_test, 
                    sampling_withoutlag_stable_lowrisk,
                    constraint=None)

In [22]:
# sample数の確認
len(df_development_unstable_train_highrisk_without_lag), len(df_development_unstable_train_lowrisk_without_lag), len(df_development_stable_train_without_lag), len(df_development_unstable_test_highrisk_without_lag), len(df_development_unstable_test_lowrisk_without_lag), len(df_development_stable_test_without_lag)

(145221, 24434, 229455, 36123, 6416, 57382)

In [23]:
# train, test dataframeの結合（lag特徴量を使わないモデル用）
df_development_train_sampled_without_lag = pl.concat([df_development_unstable_train_highrisk_without_lag, df_development_unstable_train_lowrisk_without_lag, df_development_stable_train_without_lag])
df_development_test_sampled_without_lag = pl.concat([df_development_unstable_test_highrisk_without_lag, df_development_unstable_test_lowrisk_without_lag, df_development_stable_test_without_lag])

In [24]:
# 関数を適用し時間のrandom samplingを行う（lag特徴量を使うモデル用）
df_development_unstable_train_highrisk_with_lag = sample_data(
                    df_development_unstable_train.filter(pl.col('y')==1), 
                    sampling_withlag_unstable_highrisk,
                    constraint="remove_6hr")
df_development_unstable_train_lowrisk_with_lag = sample_data(
                    df_development_unstable_train.filter(pl.col('y')==0), 
                    sampling_withlag_unstable_lowrisk,
                    constraint="remove_6hr")
df_development_stable_train_with_lag = sample_data(
                    df_development_stable_train, 
                    sampling_withlag_stable_lowrisk,
                    constraint="remove_6hr")

df_development_unstable_test_highrisk_with_lag = sample_data(
                    df_development_unstable_test.filter(pl.col('y')==1), 
                    sampling_withlag_unstable_highrisk,
                    constraint="remove_6hr")
df_development_unstable_test_lowrisk_with_lag = sample_data(
                    df_development_unstable_test.filter(pl.col('y')==0), 
                    sampling_withlag_unstable_lowrisk,
                    constraint="remove_6hr")
df_development_stable_test_with_lag = sample_data(
                    df_development_stable_test, 
                    sampling_withlag_stable_lowrisk,
                    constraint="remove_6hr")

In [25]:
# train, test dataframeの結合（lag特徴量を使うモデル用）
df_development_train_sampled_with_lag = pl.concat([df_development_unstable_train_highrisk_with_lag, df_development_unstable_train_lowrisk_with_lag, df_development_stable_train_with_lag])
df_development_test_sampled_with_lag = pl.concat([df_development_unstable_test_highrisk_with_lag, df_development_unstable_test_lowrisk_with_lag, df_development_stable_test_with_lag])

## Relabel as low risk

木下さんに相談の上、この処理をかませることに決定 @9/16

In [26]:
# モデル評価データについて、リスク不明期間をlow riskとしてrelabel

df_development_test_all = df_development_test_all.with_columns(
    pl.when(pl.col('y') == -1).then(0).otherwise(pl.col('y')).alias('y')
)

## Join all features

In [27]:
# メモリ節約
del df_development

In [28]:
# フルデータの読み込み（遅延評価）
lazy_full_df = pl.scan_csv(os.path.join(data_dir + features_file_name))

In [29]:
# ここまで計算したdfを一旦lazy frameに変換
lazy_df_development_train_sampled_baseline = df_development_train_sampled_baseline.lazy()
lazy_df_development_test_sampled_baseline = df_development_test_sampled_baseline.lazy()
lazy_df_development_train_sampled_without_lag = df_development_train_sampled_without_lag.lazy()
lazy_df_development_test_sampled_without_lag = df_development_test_sampled_without_lag.lazy()
lazy_df_development_train_sampled_with_lag = df_development_train_sampled_with_lag.lazy()
lazy_df_development_test_sampled_with_lag = df_development_test_sampled_with_lag.lazy()
lazy_df_development_test_all = df_development_test_all.lazy()

In [30]:
# icu_stay_id, time_window_index, y列のみを取り出す
lazy_df_development_train_sampled_baseline = lazy_df_development_train_sampled_baseline.select('icu_stay_id', 'time_window_index', 'y')
lazy_df_development_test_sampled_baseline = lazy_df_development_test_sampled_baseline.select('icu_stay_id', 'time_window_index', 'y')
lazy_df_development_train_sampled_without_lag = lazy_df_development_train_sampled_without_lag.select('icu_stay_id', 'time_window_index', 'y')
lazy_df_development_test_sampled_without_lag = lazy_df_development_test_sampled_without_lag.select('icu_stay_id', 'time_window_index', 'y')
lazy_df_development_train_sampled_with_lag = lazy_df_development_train_sampled_with_lag.select('icu_stay_id', 'time_window_index', 'y')
lazy_df_development_test_sampled_with_lag = lazy_df_development_test_sampled_with_lag.select('icu_stay_id', 'time_window_index', 'y')
lazy_df_development_test_all = lazy_df_development_test_all.select('icu_stay_id', 'time_window_index', 'y')

In [31]:
# 共通のkey(icu_stay_id, time_window_index)で結合
features_development_train_sampled_baseline = lazy_df_development_train_sampled_baseline.join(lazy_full_df, on=['icu_stay_id','time_window_index'], how='left')
features_development_test_sampled_baseline = lazy_df_development_test_sampled_baseline.join(lazy_full_df, on=['icu_stay_id','time_window_index'], how='left')
features_development_train_sampled_without_lag = lazy_df_development_train_sampled_without_lag.join(lazy_full_df, on=['icu_stay_id','time_window_index'], how='left')
features_development_test_sampled_without_lag = lazy_df_development_test_sampled_without_lag.join(lazy_full_df, on=['icu_stay_id','time_window_index'], how='left')
features_development_train_sampled_with_lag = lazy_df_development_train_sampled_with_lag.join(lazy_full_df, on=['icu_stay_id','time_window_index'], how='left')
features_development_test_sampled_with_lag = lazy_df_development_test_sampled_with_lag.join(lazy_full_df, on=['icu_stay_id','time_window_index'], how='left')
features_development_test_all = lazy_df_development_test_all.join(lazy_full_df, on=['icu_stay_id','time_window_index'], how='left')

## Save as CSV

In [32]:
# 遅延評価を実行し、結果を収集、csvに保存
features_development_train_sampled_baseline = features_development_train_sampled_baseline.collect()
features_development_train_sampled_baseline.write_csv(os.path.join(data_dir, df_development_train_sampled_baseline_file_name))
del features_development_train_sampled_baseline

features_development_test_sampled_baseline = features_development_test_sampled_baseline.collect()
features_development_test_sampled_baseline.write_csv(os.path.join(data_dir, df_development_test_sampled_baseline_file_name))
del features_development_test_sampled_baseline

features_development_train_sampled_without_lag = features_development_train_sampled_without_lag.collect()
features_development_train_sampled_without_lag.write_csv(os.path.join(data_dir, df_development_train_sampled_without_lag_file_name))
del features_development_train_sampled_without_lag

features_development_test_sampled_without_lag = features_development_test_sampled_without_lag.collect()
features_development_test_sampled_without_lag.write_csv(os.path.join(data_dir, df_development_test_sampled_without_lag_file_name))
del features_development_test_sampled_without_lag

features_development_train_sampled_with_lag = features_development_train_sampled_with_lag.collect()
features_development_train_sampled_with_lag.write_csv(os.path.join(data_dir, df_development_train_sampled_with_lag_file_name))
del features_development_train_sampled_with_lag

features_development_test_sampled_with_lag = features_development_test_sampled_with_lag.collect()
features_development_test_sampled_with_lag.write_csv(os.path.join(data_dir, df_development_test_sampled_with_lag_file_name))
del features_development_test_sampled_with_lag

features_development_test_all = features_development_test_all.collect()
features_development_test_all = features_development_test_all.sort(['icu_stay_id','time_window_index'])
features_development_test_all.write_csv(os.path.join(data_dir, df_development_test_all_file_name))
del features_development_test_all